In [1]:
!pip install pdfplumber tqdm pandas


In [2]:
!pip install pypdf langchain sentence-transformers faiss-cpu pandas tqdm


In [3]:
from pypdf import PdfReader

pdf_path = "science class 8.pdf"

reader = PdfReader(pdf_path)

raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() + "\n"

print(raw_text[:1500])

SCIENCE
TEXTBOOK FOR CLASS VIII
SCIENCE
2018-19

First Edition
January  2008  Magha 1929
Reprint Edition
December 2008  Pausa 1930
January  2010  Magha 1931
November 2010  Kartika 1932
January 2012  Magha 1933
November 2012  Kartika 1934
October 2013  Asvina 1935
December 2014  Pausa 1936
December 2015 Agrahayana 1937
February 2017 Magha 1938
December 2017 Agrahayana 1939
PD 750T+100T RPS
© National Council of Educational
Research and Training, 2008
` 55.00
ISBN  978-81-7450-812-6
ALL RIGHTS RESERVED
/boxshadowdwnNo part of this publication may be reproduced, stored in a retrieval
system or transmitted, in any form or by any means, electronic,
mechanical, photocopying, recording or otherwise without the prior
permission of the publisher.
/boxshadowdwnThis book is sold subject to the condition that it shall not, by way of
trade,  be lent,  re-sold, hired out or otherwise disposed of without the
publisher’s consent, in any form of binding or cover other than that in
which it is published

In [4]:
import re

def clean_text(text):
    text = re.sub(r'\n+', '\n', text)        
    text = re.sub(r'\s+', ' ', text)         
    text = re.sub(r'Page\s+\d+', '', text)   
    return text.strip()

cleaned_text = clean_text(raw_text)

print(cleaned_text[:1500])


SCIENCE TEXTBOOK FOR CLASS VIII SCIENCE 2018-19 First Edition January 2008 Magha 1929 Reprint Edition December 2008 Pausa 1930 January 2010 Magha 1931 November 2010 Kartika 1932 January 2012 Magha 1933 November 2012 Kartika 1934 October 2013 Asvina 1935 December 2014 Pausa 1936 December 2015 Agrahayana 1937 February 2017 Magha 1938 December 2017 Agrahayana 1939 PD 750T+100T RPS © National Council of Educational Research and Training, 2008 ` 55.00 ISBN 978-81-7450-812-6 ALL RIGHTS RESERVED /boxshadowdwnNo part of this publication may be reproduced, stored in a retrieval system or transmitted, in any form or by any means, electronic, mechanical, photocopying, recording or otherwise without the prior permission of the publisher. /boxshadowdwnThis book is sold subject to the condition that it shall not, by way of trade, be lent, re-sold, hired out or otherwise disposed of without the publisher’s consent, in any form of binding or cover other than that in which it is published. /boxshadowdw

In [5]:
print(cleaned_text[:3000])


SCIENCE TEXTBOOK FOR CLASS VIII SCIENCE 2018-19 First Edition January 2008 Magha 1929 Reprint Edition December 2008 Pausa 1930 January 2010 Magha 1931 November 2010 Kartika 1932 January 2012 Magha 1933 November 2012 Kartika 1934 October 2013 Asvina 1935 December 2014 Pausa 1936 December 2015 Agrahayana 1937 February 2017 Magha 1938 December 2017 Agrahayana 1939 PD 750T+100T RPS © National Council of Educational Research and Training, 2008 ` 55.00 ISBN 978-81-7450-812-6 ALL RIGHTS RESERVED /boxshadowdwnNo part of this publication may be reproduced, stored in a retrieval system or transmitted, in any form or by any means, electronic, mechanical, photocopying, recording or otherwise without the prior permission of the publisher. /boxshadowdwnThis book is sold subject to the condition that it shall not, by way of trade, be lent, re-sold, hired out or otherwise disposed of without the publisher’s consent, in any form of binding or cover other than that in which it is published. /boxshadowdw

In [6]:
chapter_titles = [
    "Crop Production and Management",
    "Microorganisms: Friend and Foe",
    "Synthetic Fibres and Plastics",
    "Materials: Metals and Non-Metals",
    "Coal and Petroleum",
    "Combustion and Flame",
    "Conservation of Plants and Animals",
    "Cell—Structure and Functions",
    "Reproduction in Animals",
    "Reaching the Age of Adolescence",
    "Force and Pressure",
    "Friction",
    "Sound",
    "Chemical Effects of Electric Current",
    "Some Natural Phenomena",
    "Light",
    "Stars and the Solar System",
    "Pollution of Air and Water"
]


In [7]:
documents = []

for i in range(len(chapter_titles) - 1):
    start = cleaned_text.find(chapter_titles[i])
    end = cleaned_text.find(chapter_titles[i+1])

    if start != -1 and end != -1:
        chapter_text = cleaned_text[start:end]
        documents.append({
            "chapter": f"Chapter {i+1}",
            "title": chapter_titles[i],
            "content": chapter_text.strip()
        })


In [15]:
len(documents)


2

In [17]:
chapter_anchors = [
    "Crop Production",
    "Microorganisms",
    "Synthetic Fibres",
    "Materials Metals",
    "Coal and Petroleum",
    "Combustion",
    "Conservation of Plants",
    "Cell Structure",
    "Reproduction in Animals",
    "Reaching the Age of Adolescence",
    "Force and Pressure",
    "Friction",
    "Sound"
]


In [19]:
normalized_text = cleaned_text.lower()


In [21]:
positions = []

for anchor in chapter_anchors:
    idx = normalized_text.find(anchor.lower())
    if idx != -1:
        positions.append(idx)

positions.sort()
len(positions)


12

In [23]:
len(documents)


2

In [25]:
!pip install -U langchain-text-splitters



In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(cleaned_text)

len(chunks) 


1135

In [27]:
!pip install -U sentence-transformers faiss-cpu


In [31]:
documents = []

for i, chunk in enumerate(chunks):
    documents.append({
        "id": i,
        "text": chunk
    })

len(documents)


1135

In [33]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


In [34]:
texts = [doc["text"] for doc in documents]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

embeddings.shape


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

(1135, 384)

In [35]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

index.ntotal


1135

In [37]:
def retrieve(query, k=10):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding, k)

    return [documents[i]["text"] for i in indices[0]]



In [38]:
context_chunks = retrieve("What is photosynthesis?", k=10)

for i, chunk in enumerate(context_chunks):
    print(f"\n--- Chunk {i+1} ---\n")
    print(chunk[:300])




--- Chunk 1 ---

for photosynthesis, thereby decreasing the amount of CO 2 in the air . Defor estation leads to an increase in the amount of CO2 in the air because the number of trees which consume CO 2 is reduced. Human activities, thus, contribute to the accumulation of CO2 in the atmosphere. CO2 traps heat and do

--- Chunk 2 ---

leaf. They ar e scattered in the cytoplasm of the leaf cells. These are called plastids. They are of different colours. Some of them contain green pigment called chlorophyll. Green coloured plastids are called chloroplasts. They provide green colour to the leaves. You may recall that chlorophyll in 

--- Chunk 3 ---

function. 8. ‘Cells are the basic structural units of living organisms’. Explain. 9. Explain why chloroplasts are found only in plant cells? 10. Complete the crossword with the help of clues given below. Across 1. This is necessary for photosynthesis. 3. Term for component present in the cytoplasm. 

--- Chunk 4 ---

r ead that the sun produc

In [39]:
!pip install ollama


In [40]:
import ollama

response = ollama.chat(
    model="gemma:2b",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response["message"]["content"])


Hello! 👋 It's a pleasure to meet you as well. How can I help you today?


In [47]:
def rag_answer(question, k=20):
    retrieved_chunks = retrieve(question, k)

    # Combine retrieved text
    combined_context = " ".join([chunk[0].lower() for chunk in retrieved_chunks])

    # HARD CHECK: if keyword not in context, refuse
    key_terms = ["photosynthesis"]

    if not any(term in combined_context for term in key_terms):
        return "I’m focused on Class 8 Science. The provided textbook context does not contain the answer. Please ask a related question."

    context = "\n\n".join(
        [f"[Source {i+1}]\n{chunk[0]}" for i, chunk in enumerate(retrieved_chunks)]
    )

    system_prompt = (
    "You are an AI tutor for NCERT Class 8 Science.\n"
    "Answer the question using ONLY the provided textbook context.\n"
    "You may combine information from multiple sources to form a complete answer.\n"
    "Use simple, grade-appropriate language.\n"
    "If the context contains related information, explain it clearly.\n"
    "ONLY say you cannot answer if the topic is completely unrelated to Class 8 Science.\n"
    "Always cite sources like (Source 1), (Source 2)."
 )


    response = ollama.chat(
        model="gemma:2b",
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ],
        options={"temperature": 0.1}
    )

    return response["message"]["content"]



In [49]:
print(rag_answer("What is photosynthesis?"))


I’m focused on Class 8 Science. The provided textbook context does not contain the answer. Please ask a related question.


In [51]:
matches = [doc for doc in documents if "photosynthesis" in doc["text"].lower()]
len(matches)


5

In [53]:
def retrieve(query, k=5):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding, k)

    return [(documents[i]["text"], distances[0][idx]) for idx, i in enumerate(indices[0])]


In [55]:
results = retrieve("What is photosynthesis?", k=3)

for res in results:
    print(res[0][:300])
    print("------")


for photosynthesis, thereby decreasing the amount of CO 2 in the air . Defor estation leads to an increase in the amount of CO2 in the air because the number of trees which consume CO 2 is reduced. Human activities, thus, contribute to the accumulation of CO2 in the atmosphere. CO2 traps heat and do
------
leaf. They ar e scattered in the cytoplasm of the leaf cells. These are called plastids. They are of different colours. Some of them contain green pigment called chlorophyll. Green coloured plastids are called chloroplasts. They provide green colour to the leaves. You may recall that chlorophyll in 
------
function. 8. ‘Cells are the basic structural units of living organisms’. Explain. 9. Explain why chloroplasts are found only in plant cells? 10. Complete the crossword with the help of clues given below. Across 1. This is necessary for photosynthesis. 3. Term for component present in the cytoplasm. 
------


In [57]:
results = retrieve("photosynthesis", k=10)

for i, (text, dist) in enumerate(results, 1):
    print(f"\n--- Chunk {i} ---")
    print(text[:400])



--- Chunk 1 ---
for photosynthesis, thereby decreasing the amount of CO 2 in the air . Defor estation leads to an increase in the amount of CO2 in the air because the number of trees which consume CO 2 is reduced. Human activities, thus, contribute to the accumulation of CO2 in the atmosphere. CO2 traps heat and does not allow it to escape into space. As a result, the average temperature of the earth’s atmosphere

--- Chunk 2 ---
leaf. They ar e scattered in the cytoplasm of the leaf cells. These are called plastids. They are of different colours. Some of them contain green pigment called chlorophyll. Green coloured plastids are called chloroplasts. They provide green colour to the leaves. You may recall that chlorophyll in the chloroplasts of leaves, is essential for photosynthesis. 8.6 Comparison of Plant and Animal Cell

--- Chunk 3 ---
by it Fig. 18.4 : Taj Mahal 2018-19 POLLUTION OF AIR AND WATER 243 the role of carbon dioxide in plants. But if ther e is excess of CO 2 in the air

In [59]:
def rag_answer(question, k=5):
    retrieved_chunks = retrieve(question, k)

    context = "\n\n".join(
        [f"[Source {i+1}]\n{chunk[0]}" for i, chunk in enumerate(retrieved_chunks)]
    )

    system_prompt = (
        "You are an AI tutor for NCERT Class 8 Science.\n"
        "Answer using ONLY the provided textbook sources.\n"
        "You may combine information from multiple sources.\n"
        "Use simple, grade-appropriate language.\n"
        "If the topic is completely NOT present in the sources, say:\n"
        "'I’m focused on Class 8 Science. Please ask a related question.'\n"
        "Always cite sources like (Source 1), (Source 2)."
    )

    response = ollama.chat(
        model="gemma:2b",
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ],
    )

    return response["message"]["content"]


In [61]:
print(rag_answer("What is photosynthesis?"))


Sure. Here's the definition of photosynthesis from the textbook:

**Photosynthesis** is a biological process in which plants and certain other organisms use the energy from sunlight to convert carbon dioxide and water into sugar and oxygen.


In [62]:
print(rag_answer("What is sound?"))


According to the textbook sources, sound is the movement of a sound wave through a medium. Sound waves are a disturbance in the medium that creates a sound. These waves can be produced by a variety of sources, including musical instruments, machines, and natural phenomena.

Sound is often unpleasant when it is too loud, but it can also be pleasing when it is harmonious and well-tuned. The frequency, amplitude, and other characteristics of a sound wave determine its pitch, loudness, and other qualities.

According to the textbook sources, sound is a complex phenomenon that requires an understanding of physics and acoustics to fully comprehend.


In [65]:
import pandas as pd

data = []

questions = [
    "What is photosynthesis?",
    "What is chlorophyll?",
    "Why are chloroplasts found only in plant cells?",
    # add more
]

for q in questions:
    ans = rag_answer(q)
    data.append({"question": q, "answer": ans})

df = pd.DataFrame(data)
df.to_csv("evaluation.csv", index=False)
